# Evidence, notes and handoffs

Rewrite a poor helpdesk ticket into a triage note, an escalation and a shift handover that the next analyst can act on, using the evidence from a real phishing case.

The files for this seminar are in `~/case` (Linux) and `case/` (Python): `alerts.json`, `handover-night.txt`, `invoice.eml`, `mail.log`, `process.csv`, `proxy.log`, `share_listing.txt`, `ticket-88213.txt`.

## 1. A ticket nobody can act on

Most security work is handed from one person to the next: helpdesk to SOC, SOC to the incident lead, night shift to morning shift. If the note is poor, the next person starts again from zero, and an attack keeps running while they do.

Read helpdesk ticket 88213 and last night's handover. Both are real kinds of notes analysts receive every day.

- [ ] Which facts in the ticket have a time and a source?
- [ ] Which words are guesses ("probably", "maybe", "???")?
- [ ] What does the next person need that isn't there: which host, which user, which alert, what was done?
- [ ] What is wrong with "all quiet. couple of alerts, nothing major"?

> Hint: A useful note answers five questions: what fired, what I checked, what I found, what I don't know, and what happens next.

?> Which message ID did dev.kapoor report as phishing? :: sha256:3b4c868aaed95ad3038a9cf3f9d3aac10231f741a5500448dc1f2bba9fbbc5ae
?> At what time (HH:MM) did the helpdesk pull the cable on the HR PC? :: sha256:fdf6b948b07448b5dad617e71a41f1a1fb3a25b5f9ac5e2e2adf48410cd1f02a

In [ ]:
%%bash
cd ~/case
cat ticket-88213.txt
echo ---
cat handover-night.txt

## 2. Find the facts behind the ticket

A good note is built from evidence, not from what someone said on the phone. Start with the reported message in the mail log, then see who downloaded the attachment.

`nrcpt=` is the number of people the email went to. In the proxy log, `TCP_MISS/200` means the download worked and `TCP_DENIED/403` means the proxy blocked it.

- [ ] How many people received the email, and who are they?
- [ ] Which internal address downloaded the invoice, and was anyone blocked?
- [ ] Write each fact down with its time and the file you saw it in

> Hint: Proxy times are Unix epoch seconds. The Python cell turns them into UTC clock times, so every fact in your note uses one time zone.

?> How many people received the fake invoice? :: sha256:4e07408562bedb8b60ce05c1decfe3ad16b72230967de01f640b7e4729b49fce
?> Which internal IP address downloaded the invoice successfully? :: sha256:ecda40e9a719dbe5a19ca890028508dc0dffab4ef41d0accd82d2c3a46a8405d

In [ ]:
%%bash
cd ~/case
grep 4A1F21C02 mail.log
echo ---
grep -E 'report|blocklist' mail.log
echo ---
grep -E 'docm|s2.bin' proxy.log

In [ ]:
from datetime import datetime, timezone

for line in open("case/proxy.log"):
    f = line.split()
    if "docm" in f[6] or "s2.bin" in f[6]:
        when = datetime.fromtimestamp(float(f[0]), timezone.utc).strftime("%H:%M:%S UTC")
        print(when, f[2], f[3], f[6])

## 3. What ran on the PC

"HR PC slow, maybe virus" is a guess. The process list says exactly what ran, when and which program started it. Follow the `ppid` column: each process names its parent.

- [ ] Which program started cmd.exe? Should it ever do that?
- [ ] Where does the program PowerShell started live? Where does the real svchost.exe live?
- [ ] When did that program reach the file share?

> Hint: The real svchost.exe is in C:\Windows\System32 and runs as SYSTEM. A file called svchost-upd.exe in C:\Users\Public is pretending.

?> Which program started cmd.exe? (file name) :: sha256:f8878dc09ca0bd8005c9009ae3a4aee3085d3081b5c95e2232c9d8e561ce9166
?> What is the PID of svchost-upd.exe started by PowerShell? :: sha256:cfe35c9e498096d4463cfade00cdef305d2311d90db9949bf321f87447fc2f1a

In [ ]:
import csv
from collections import defaultdict

rows = [r for r in csv.DictReader(open("case/process.csv")) if r["host"] == "hr-pc07"]
kids = defaultdict(list)
for r in rows:
    kids[r["ppid"]].append(r)


def show(pid, depth):
    for r in kids.get(pid, []):
        print("    " * depth + r["time"][11:] + " UTC  " + r["image"].split("\\")[-1] + "  (pid " + r["pid"] + ")  " + r["command_line"][:60])
        show(r["pid"], depth + 1)


show("704", 0)

## 4. Write the triage note

Now rewrite ticket 88213 as a triage note for alert **AL-2203**. The report builder checks each section as you type: it wants an alert ID, times with a time zone, the exact searches you ran, at least one gap, a disposition and an owner.

Rules that make notes useful:

- One time zone for everything (UTC here), written every time.
- Facts say where you saw them. Guesses go under "What I don't know yet".
- Write what you ran, so the next person can repeat it without asking you.

- [ ] Fill every section, then press Check the report
- [ ] Compare yours with the model answer only after you've written your own
- [ ] Download the Markdown: it is part of your hand-in

> Hint: "What I don't know yet" is never empty. Here: did the other two people open the invoice, and was anything copied out?

```tool report
{"tpl": "triage", "title": "Triage note: AL-2203 macro document on hr-pc07", "text": "# Triage note: AL-2203 macro document on hr-pc07\n\n## What fired\n\nAL-2203 \"Office document with macros opened from the internet\", severity high, host hr-pc07, user priya.nair, 2026-11-02 09:02:44 UTC.\n\n## What I checked\n\n- mail.log: grep 4A1F21C02 (the reported message)\n- proxy.log: grep -E 'docm|s2.bin'\n- process.csv: the process tree for hr-pc07 (Python cell, step 3)\n- alerts.json: the other alerts on hr-pc07 between 09:00 and 10:00 UTC\n\n## What I found\n\n- 08:51 UTC: message 4A1F21C02 from bounce-7781@kestrel-logistlcs.com (look-alike domain, 203.0.113.45) was delivered to 3 people: priya.nair, dev.kapoor, tomas.reyes.\n- 09:02 UTC: 10.20.0.14 downloaded INV-30412.docm from kestrel-logistlcs.com (proxy TCP_MISS/200).\n- 09:02:55 UTC: WINWORD.EXE started cmd.exe, which started a hidden PowerShell running C:\\Users\\Public\\upd.ps1.\n- 09:03:09 UTC: hr-pc07 downloaded s2.bin from cdn-update-check.net (198.51.100.140).\n- 09:03:20 UTC: PowerShell started C:\\Users\\Public\\svchost-upd.exe (PID 4810).\n- 09:41 UTC: PID 4810 mapped \\\\fs01\\shared and started renaming files to .kestrel-locked (AL-2208).\n\n## What I don't know yet\n\n- Whether dev.kapoor or tomas.reyes opened the attachment (10.20.0.15 was blocked at 09:20 UTC; the third person is unknown).\n- What s2.bin does, and whether data was copied out before the encryption.\n- Whether other hosts talked to 198.51.100.140.\n\n## Disposition and why\n\nTrue positive, escalate: a macro document from a look-alike domain started PowerShell and an unknown program, and the same program is encrypting the file share.\n\n## Next step and owner\n\nEscalate to the incident lead now (09:55 UTC). Owner: SOC analyst on shift until the incident lead takes it; keep hr-pc07 off the network and do not reboot it.\n", "n": 6, "blank": ["text"]}
```

?> Which disposition fits AL-2203? (escalate, handled, benign, false positive or more information) :: sha256:f965e5fbf98dd531b7c0fd4f9bdc8dd6df79f623c8345b8aa1b7cd756d317dd8

## 5. Escalate

An escalation is read in ten seconds by someone who has just been woken up. Put **the ask** in the first line: what you need, from whom, by when. Then severity, what you know, what you don't know, impact, what you did, and where the evidence is.

Say what you have **not** done as well: "I have not changed anything on the host" tells the incident lead the evidence is untouched.

- [ ] First line: the ask, with a time
- [ ] Severity, and the one fact that decided it
- [ ] Facts with times, newest last
- [ ] Where the evidence is saved (file names or hashes)

> Hint: Critical means harm is happening now. Files being encrypted on a shared drive is the fact that decides it.

```tool report
{"tpl": "escalation", "title": "Escalation: ransomware on hr-pc07 and fs01", "text": "# Escalation: ransomware on hr-pc07 and fs01\n\n## The ask\n\nPlease open a critical incident and approve isolating hr-pc07 and taking the HR and Finance shares on fs01 read-only, by 10:05 UTC. Incident lead and IT operations, please call the SOC.\n\n## Severity and why\n\nCritical: files on fs01 are being encrypted (AL-2208), so business data is being lost right now.\n\n## What we know\n\n- 08:51 UTC: a fake invoice from kestrel-logistlcs.com reached 3 people (mail.log, message 4A1F21C02).\n- 09:02 UTC: priya.nair on hr-pc07 opened INV-30412.docm; Word started cmd.exe and a hidden PowerShell (process.csv).\n- 09:03 UTC: PowerShell downloaded s2.bin from cdn-update-check.net and started svchost-upd.exe from C:\\Users\\Public (proxy.log, process.csv).\n- 09:04 UTC: a new local administrator, support_svc, was created on hr-pc07 (AL-2210).\n- 09:41 to 09:50 UTC: files on \\\\fs01\\shared\\HR and \\Finance were renamed to .kestrel-locked (share_listing.txt).\n- 09:52 UTC: the helpdesk pulled the network cable on hr-pc07 (ticket 88213).\n\n## What we don't know\n\n- Whether the other two recipients opened the invoice.\n- Whether any data was copied out before the encryption.\n- Whether the attacker has access to any other host.\n\n## Impact so far\n\nHost hr-pc07 and user account priya.nair are compromised. 10 files on the HR and Finance shares are encrypted, including payroll and vendor bank details (personal data). Ops is not encrypted. The encryption stopped after the cable was pulled.\n\n## What I've done\n\n- 09:52 UTC: the helpdesk pulled the cable on hr-pc07 (not me; I have not changed anything on the host).\n- 09:55 UTC: I asked the proxy team to confirm kestrel-logistlcs.com and cdn-update-check.net are blocked.\n\n## Evidence\n\nSaved in the incident folder with SHA-256 hashes: alerts.json, mail.log, proxy.log, process.csv, share_listing.txt and invoice.eml. Ticket 88213 holds the helpdesk notes.\n", "n": 7, "blank": ["text"]}
```

?> How many files on the share were encrypted? :: sha256:4a44dc15364204a80fe80e9039455cc1608281820fe2b24f1e5233ade6af1dd5
?> Which folder on the share was not encrypted? :: sha256:a92c36e66a25ee99ff862faa8e87987be6c7cd13c3ee661c400a45b0f1e3b132

In [ ]:
%%bash
cd ~/case
grep -c 'kestrel-locked' share_listing.txt
grep 'Directory of' share_listing.txt
echo ---
sha256sum alerts.json mail.log proxy.log process.csv share_listing.txt invoice.eml

## 6. Work the queue before you hand over

Your shift had ten alerts, and the night shift left a note that says "nothing major". Give every alert a disposition and a one-line reason. Some belong to the incident; some are ordinary noise and should be closed so the next shift isn't chasing them.

The alert queue sorts by severity, then time. Pick an alert, choose a disposition, and write why.

- [ ] Every alert gets a disposition: nothing left "Not triaged"
- [ ] Each reason is one line a stranger could check
- [ ] Write the notes for the next shift in the box under the queue
- [ ] Download the handover the queue writes for you

> Hint: AL-2202's "Singapore" address is the company VPN exit: read the detail line. AL-2206 retried and worked at 09:25.

```tool alerts
{"file": "alerts.json", "d": {"AL-2201": "escalate", "AL-2202": "benign", "AL-2203": "escalate", "AL-2204": "escalate", "AL-2205": "escalate", "AL-2210": "escalate", "AL-2206": "benign", "AL-2209": "benign", "AL-2208": "escalate", "AL-2207": "escalate"}, "n": 7, "blank": ["d"]}
```

?> Which alert ID is the sign-in from the company VPN exit? :: sha256:a7b16ecff6a60f14396ba3105151e8473eb42064720f436cd3adef6a9720afc4
?> Which local account did the attacker create on hr-pc07? :: sha256:bd3adca4175f142eaabf87ce2580309d9b03835634a38a8654216c9ec751e322

## 7. Write the shift handover

Rewrite last night's "all quiet" note as a handover that leaves nothing behind. List every open item **by ID** with the next step and who owns it, what you closed, what to watch, who you called and when, and any temporary change that must be undone later.

- [ ] Every open item by ID, with next step and owner
- [ ] Closed items in one line each
- [ ] Things to watch on the next shift
- [ ] Calls and messages with times
- [ ] Temporary blocks or exceptions, with a review date

> Hint: A temporary block nobody remembers becomes a permanent one. Write down who may remove it, and when it should be reviewed.

```tool report
{"tpl": "handover", "title": "Shift handover: morning to afternoon, 02 Nov", "text": "# Shift handover: morning to afternoon, 02 Nov\n\n## Still open\n\n- INC-0211 (from AL-2203, AL-2204, AL-2205, AL-2208, AL-2210): ransomware on hr-pc07 and fs01. Contained at 09:52 UTC. Next: incident lead decides on restore from snapshot s-1101; IT ops owns it.\n- AL-2201: same campaign. Next: confirm whether dev.kapoor and tomas.reyes opened the invoice. Owner: afternoon analyst, by 14:00 UTC.\n- AL-2207: 6 failed network logons from 198.51.100.140 at 09:59 UTC for support_svc and Administrator. Next: confirm the account support_svc is disabled. Owner: IT ops.\n\n## Closed this shift\n\n- AL-2202: benign, the Singapore address is the company VPN exit.\n- AL-2206: benign, the antivirus update retried and worked at 09:25 UTC.\n- AL-2209: benign, a large print job on the second floor printer.\n\n## Watch for\n\n- Any connection from any host to 198.51.100.140 or cdn-update-check.net.\n- Any new account named support_svc on another host.\n- New files ending in .kestrel-locked on any share.\n\n## Calls and messages\n\n- 09:55 UTC: escalated to the incident lead by phone; answered, incident INC-0211 opened.\n- 10:02 UTC: messaged IT ops about the share; answered at 10:06 UTC.\n\n## Tools and access\n\nTemporary proxy block on kestrel-logistlcs.com and cdn-update-check.net added at 09:18 and 09:56 UTC. Review it on 09 Nov; do not remove it without the incident lead.\n", "n": 5, "blank": ["text"]}
```

## 8. Hand in

Hand in three pieces of writing. Your teacher scores them with the rubric below, and your certificate is issued after the hand-in is scored.

| Piece | What it must have |
|---|---|
| Triage note for AL-2203 | Alert ID, time with time zone, searches you ran, facts with sources, gaps, disposition, owner |
| Escalation | The ask first, severity, know / don't know, impact, actions, evidence location |
| Shift handover | Open items by ID, closed items, watch list, calls with times, temporary changes |

Use **Hand in** at the top of the notebook. Your notes, tool work and answers go with it.

- [ ] All questions in this notebook answered
- [ ] Triage note, escalation and handover each show every section complete
- [ ] Read each one aloud once: if a sentence needs you to explain it, rewrite it
- [ ] Press Hand in

**Rubric**

| Criterion | Points |
|---|---|
| Triage note: alert ID, time with time zone, and what was checked, so someone else can repeat it | 4 |
| Triage note: facts with times and sources, kept apart from guesses | 4 |
| Triage note: a disposition with one sentence of reasons | 3 |
| Escalation: the ask in the first line, with who and by when | 3 |
| Escalation: what we know, what we don't know, and impact | 4 |
| Escalation: where the evidence is saved | 2 |
| Shift handover: every open item by ID, with the next step and owner | 4 |
| Shift handover: things to watch and calls made, with times | 3 |
| Plain writing: short sentences, no blame, no guessing presented as fact | 3 |